# Analiza dialogowa scenariusza: z jakim ładunkiem emocjonalnym?

## Wersja studencka — moduł 3: EMOCJE

W poprzednim module wydobyłeś kwestie dialogowe i zmierzyłeś, *kto* mówi i *ile*. Teraz pytamy o *jak* — z jakim ładunkiem emocjonalnym. Do oceny emocji użyjemy modelu językowego (Gemini), który przeanalizuje każdą wypowiedź i przypisze jej walencję (od −1 do +1) oraz emocję dyskretną (złość, radość, smutek itd.).

Efektem końcowym będą:
1. wczytanie kwestii z `dialogi.csv` i danych o postaciach z `postacie.csv`,
2. konfiguracja połączenia z Gemini,
3. batchowa ocena emocji wszystkich kwestii przez model,
4. łuk emocjonalny filmu (trajektoria walencji wzdłuż scen),
5. wykres rozkładu emocji według płci postaci,
6. zapis pliku `emocje.csv` do dalszej analizy.

## Jak pracować z tym notatnikiem

1. Upewnij się, że masz w tym samym środowisku pliki `dialogi.csv` i `postacie.csv` z poprzedniego modułu.
2. Skopiuj prompt z bieżącego kroku do modelu AI.
3. Wklej otrzymany kod do pustej komórki pod promptem.
4. Uruchom kod i porównaj rezultat z sekcją **Po uruchomieniu powinieneś zobaczyć**.
5. Jeśli wynik nie zgadza się z opisem, popraw tylko bieżący krok.
6. Dopiero po uzyskaniu poprawnego wyniku przejdź dalej.

**Uwaga do kroku z API:** w Etapie 2 będziesz potrzebować klucza do Gemini API. Możesz go wygenerować bezpłatnie na stronie Google AI Studio (makersuite.google.com). Klucz jest prywatny — nie wklejaj go na stałe do komórek, które zamierzasz udostępniać.

---
## Etap 1: Wczytanie danych wejściowych

Zaczynamy od danych, które wyprodukowałeś w poprzednim module.

### Krok 1A. Wczytanie `dialogi.csv` i `postacie.csv`

#### Cel i sens analityczny

`dialogi.csv` zawiera po jednym wierszu na kwestię dialogową (scena, postać, tekst, liczba słów). `postacie.csv` zawiera podsumowanie postaci z przypisaną płcią. Musimy załadować oba pliki i upewnić się, że wyglądają poprawnie.

#### Prompt dla modelu

```text
Kontekst:
Masz dwa pliki CSV z poprzedniego modułu analizy dialogowej scenariusza.

Wejście:
Pliki `dialogi.csv` (kolumny: numer sceny, postać, treść kwestii, liczba słów) i `postacie.csv` (kolumny: postać, płeć, liczba słów, liczba kwestii, liczba scen). Oba pliki są w kodowaniu UTF-8.

Zadanie:
Wczytaj oba pliki do tabel i wyświetl krótki podgląd każdego z nich.

Pokaż wynik:
- liczbę wierszy i nazwy kolumn każdego pliku,
- pierwsze 5 wierszy z `dialogi.csv`,
- wszystkie wiersze z `postacie.csv` (jest ich mało).

Warunek poprawności:
Kolumny powinny odpowiadać opisanemu formatowi. Jeśli plik ma inne nazwy kolumn niż oczekiwane, wypisz, co faktycznie znalazłeś.

Jeśli wystąpi błąd:
Wyjaśnij, którego pliku nie udało się wczytać i z jakiego powodu (zła ścieżka, kodowanie, format).

Nie rób jeszcze:
Nie oceniaj emocji.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Liczbę wierszy i kolumny obu plików.
- Pierwsze wiersze z `dialogi.csv` wyglądające jak wpisy kwestii dialogowych.
- Tabelę postaci z kolumną płeć zawierającą wartości K, M lub ?.

---
## Etap 2: Konfiguracja modelu językowego

W tym etapie instalujemy bibliotekę Gemini, podłączamy klucz API i sprawdzamy, czy połączenie działa.

### Krok 2A. Instalacja i klucz API

#### Cel i sens analityczny

Żeby wywołać Gemini z kodu Pythona, potrzebna jest biblioteka `google-genai` i klucz API. Klucz wpisujemy przez `getpass` — tak, żeby nie pojawił się w historii komórek.

#### Prompt dla modelu

```text
Kontekst:
Pracujesz w Google Colab i chcesz wywołać Gemini API z Pythona.

Wejście:
Brak — konfiguracja od zera.

Zadanie:
Zainstaluj pakiet `google-genai`. Następnie poproś użytkownika o wpisanie klucza API przez `getpass.getpass` i skonfiguruj klienta Gemini. Na końcu wyślij jedno testowe żądanie (krótki tekst: „Hello!") do modelu `gemini-2.0-flash` i wydrukuj odpowiedź, żeby potwierdzić że połączenie działa.

Pokaż wynik:
- komunikat, że instalacja się powiodła,
- pole do wpisania klucza (bez jego wyświetlania),
- krótka odpowiedź Gemini na testowe zapytanie.

Warunek poprawności:
Model musi odpowiedzieć — nawet jednym zdaniem. Brak odpowiedzi lub błąd 403/401 oznacza problem z kluczem.

Jeśli wystąpi błąd:
Wyświetl treść błędu i podpowiedź: sprawdź, czy klucz jest aktywny i czy nie przekroczyłeś limitu darmowego tieru.

Nie rób jeszcze:
Nie oceniaj kwestii dialogowych.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Potwierdzenie instalacji i pole do wpisania klucza API.
- Krótką odpowiedź Gemini potwierdzającą, że połączenie działa.

---
## Etap 3: Ocena emocji dialogów

To kluczowy etap modułu. Model językowy przeanalizuje każdą kwestię i przypisze jej:
- **walencję** — liczbę od −1 (bardzo negatywna) do +1 (bardzo pozytywna),
- **emocję dyskretną** — jedną z sześciu kategorii: `radość`, `smutek`, `złość`, `strach`, `zaskoczenie`, `neutralna`.

Żeby nie wyczerpać darmowego limitu Gemini, wysyłamy kwestie **partiami** (ang. *batch*) — po 30 na raz.

### Krok 3A. Batchowa ocena emocji

#### Cel i sens analityczny

Ocena linii po linii (jedno żądanie = jedna kwestia) jest wolna i kosztowna. Batching — przesyłanie wielu kwestii w jednym żądaniu i odebranie odpowiedzi jako tablicy JSON — jest znacznie wydajniejszy. Model dostaje kontekst (numer sceny i nagłówek), żeby nie oceniać kwestii w próżni.

#### Prompt dla modelu

```text
Kontekst:
Masz tabelę kwestii dialogowych z kolumnami: numer sceny, postać, treść kwestii, liczba słów. Masz dostęp do Gemini API (klient skonfigurowany w poprzednim kroku).

Wejście:
Tabela kwestii z Kroku 1A.

Zadanie:
Napisz funkcję, która przetwarza kwestie partiami po 30. Dla każdej partii wysyła do Gemini jedno żądanie z następującą instrukcją (wbuduj ją na stałe w kod):

---
Oceń emocje poniższych kwestii dialogowych ze scenariusza filmowego.
Dla każdej kwestii zwróć obiekt JSON z polami:
- "id": numer porządkowy kwestii w tej partii (0-indeksowany),
- "walencja": liczba od -1.0 (bardzo negatywna) do 1.0 (bardzo pozytywna),
- "emocja": jedna z: radość, smutek, złość, strach, zaskoczenie, neutralna.

Zwróć WYŁĄCZNIE tablicę JSON, bez żadnego dodatkowego tekstu ani formatowania markdown.

Kwestie do oceny:
[TUTAJ WSTAW KWESTIE W FORMACIE: "id | postać: treść kwestii"]
---

Po odebraniu odpowiedzi sparsuj JSON i dołącz wyniki (walencja, emocja) do tabeli kwestii jako nowe kolumny. Jeśli parsowanie konkretnej partii się nie powiedzie, wypełnij brakujące wiersze wartościami: walencja=0, emocja="neutralna" i wypisz ostrzeżenie. Dodaj krótkie opóźnienie (1 sekunda) między partiami, żeby nie przekraczać limitu żądań darmowego tieru.

Pokaż wynik:
- pasek postępu lub komunikaty pokazujące, która partia jest właśnie przetwarzana,
- po zakończeniu: łączną liczbę ocenionych kwestii i próbkę 10 wierszy z kolumnami postać | treść kwestii (pierwsze 60 znaków) | walencja | emocja.

Warunek poprawności:
Liczba ocenionych kwestii powinna odpowiadać liczbie wierszy w tabeli. Kolumna emocja powinna zawierać wyłącznie wartości z listy: radość, smutek, złość, strach, zaskoczenie, neutralna.

Jeśli wystąpi błąd:
Wypisz, przy której partii wystąpił problem, pokaż surową odpowiedź Gemini i kontynuuj z pozostałymi partiami.

Nie rób jeszcze:
Nie rysuj wykresów.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Komunikaty o przetwarzaniu kolejnych partii.
- Próbkę kwestii z przypisanymi walencjami i emocjami.
- Brak pustych wartości w kolumnach walencja i emocja.

> **Wskazówka:** jeśli Gemini zwraca JSON opakowany w znaczniki ` ```json ... ``` `, poproś model o poprawkę: *"usuń markdown z odpowiedzi — zwracaj tylko czystą tablicę JSON"*.

### Krok 3B. Podgląd i kontrola jakości

#### Cel i sens analityczny

Zanim narysujemy wykresy, sprawdzamy, czy rozkład emocji wygląda sensownie — czy model nie wpadł w schemat (np. wszystko „neutralna") albo nie pominął żadnej kategorii.

#### Prompt dla modelu

```text
Kontekst:
Masz tabelę kwestii z przypisanymi emocjami i walencjami.

Wejście:
Tabela kwestii z kolumnami emocja i walencja.

Zadanie:
Pokaż podsumowanie kontrolne: rozkład sześciu kategorii emocji (ile kwestii i jaki procent), rozkład walencji (histogram albo percentyle: min, Q1, mediana, Q3, max) oraz 3 przykładowe kwestie z największą walencją i 3 z najmniejszą.

Pokaż wynik:
- tabelę: emocja | liczba kwestii | procent,
- podstawowe statystyki walencji,
- 6 przykładowych kwestii (3 najradośniejsze, 3 najsmutniejsze/najbardziej negatywne).

Warunek poprawności:
Żadna emocja nie powinna dominować z udziałem powyżej 80%. Jeśli tak jest, wypisz ostrzeżenie i zasugeruj, że prompt może wymagać korekty.

Jeśli wystąpi błąd:
Wyjaśnij, czy kolumna emocja jest pusta albo zawiera nieoczekiwane wartości.

Nie rób jeszcze:
Nie rysuj wykresów.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Tabelę z rozkładem sześciu emocji i ich procentowym udziałem.
- Statystyki walencji pokazujące rozpiętość od negatywnej do pozytywnej.
- Przykłady kwestii z dwóch biegunów emocjonalnych.

> **Jeśli rozkład jest zdegenerowany** (np. 90% „neutralna") — wróć do Kroku 3A i poproś model o zmodyfikowanie promptu do Gemini, żeby był bardziej rozróżniający. To normalna iteracja w vibe codingu.

---
## Etap 4: Łuk emocjonalny filmu

Teraz patrzymy na emocje w czasie — jak zmienia się nastrój scenariusza od pierwszej do ostatniej sceny. To tzw. *emotional arc*, znany z badań Reagana i in. (2016) oraz Vonneguta, który twierdził, że każda historia ma swój emocjonalny kształt.

### Krok 4A. Wykres łuku emocjonalnego

#### Cel i sens analityczny

Zamiast patrzeć na pojedyncze kwestie, uśredniamy walencję w ruchomym oknie scen (np. 5-scenowym). Wygładzony wykres pokazuje, gdzie film jest napięty i mroczny, a gdzie emocje się rozładowują — i czy te zmiany pokrywają się z ważnymi punktami fabularnymi.

#### Prompt dla modelu

```text
Kontekst:
Masz tabelę kwestii z kolumnami: numer sceny i walencja. Chcesz zobaczyć, jak zmienia się nastrój scenariusza w czasie.

Wejście:
Tabela kwestii z ocenionymi walencjami.

Zadanie:
Oblicz średnią walencję per scena (uśrednij walencje wszystkich kwestii w danej scenie). Następnie wygładź wynik ruchomym oknem o szerokości 5 scen. Narysuj wykres liniowy: oś X to numer sceny, oś Y to wygładzona walencja. Zaznacz poziomą linię walencji = 0 (granica pozytywne/negatywne). Jeśli znasz tytuł filmu, dodaj do wykresu 2–3 etykiety tekstowe wskazujące sceny o szczególnie wysokiej lub niskiej walencji i krótko opisujące, co w nich się dzieje.

Pokaż wynik:
- jeden wykres liniowy łuku emocjonalnego z tytułem i opisanymi osiami,
- poziomą linię zerową,
- opcjonalne etykiety przy szczytach i dolinach.

Warunek poprawności:
Wykres nie powinien być płaski (walencja stale blisko 0) — jeśli tak wygląda, wypisz ostrzeżenie, że może to oznaczać zdegenerowany wynik oceny emocji.

Jeśli wystąpi błąd:
Wyjaśnij, czy tabela kwestii nie ma kolumny walencja albo czy wszystkie numery scen są identyczne.

Nie rób jeszcze:
Nie rysuj wykresu emocja × płeć.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Wykres liniowy z trajektorią emocjonalną od pierwszej do ostatniej sceny.
- Wyraźne wzloty i upadki nastroju (chyba że film jest wyjątkowo monotonny emocjonalnie).
- Poziomą linię zerową oddzielającą nastrój pozytywny od negatywnego.

#### Pytanie interpretacyjne

Gdzie na wykresie są doliny (najbardziej mroczne sceny) i szczyty (ulgi lub radości)? Czy pokrywają się z ważnymi zwrotami fabuły, które pamiętasz z filmu? Vonnegut opisywał opowieści jako „kształty" — jaki kształt ma Twój film?

---
## Etap 5: Emocje według płci

Teraz łączymy dwa wymiary: emocje i płeć. Pytamy, czy kobiety i mężczyźni w tym scenariuszu wyrażają inne emocje — i czy rozkład jest symetryczny, czy asymetryczny.

### Krok 5A. Wykres rozkładu emocji według płci

#### Cel i sens analityczny

Kino przez dekady utrwalało stereotypy emocjonalne: kobiety płaczą, mężczyźni się złoszczą. Sieciowa analiza danych pozwala sprawdzić, czy i w jakim stopniu konkretny scenariusz reprodukuje te wzorce. Pamiętaj jednak o granicy metody: to, co mówi postać, nie musi odzwierciedlać jej głębszych emocji w narracji.

#### Prompt dla modelu

```text
Kontekst:
Masz tabelę kwestii z kolumnami emocja i postać. Masz też tabelę postacie.csv z kolumną płeć (K, M lub ?).

Wejście:
Tabela kwestii i tabela postaci.

Zadanie:
Połącz obie tabele tak, żeby każda kwestia miała przypisaną płeć postaci, która ją wypowiada. Ogranicz analizę do płci K i M (pomiń ?). Narysuj pogrupowany lub ułożony wykres słupkowy pokazujący, jaki procent kwestii danej płci należy do każdej z sześciu kategorii emocji. Użyj znormalizowanych wartości procentowych (a nie bezwzględnych), żeby wyrównać ewentualną różnicę w ogólnej liczbie kwestii K i M. Dodaj tytuł, opis osi i legendę.

Pokaż wynik:
- jeden wykres z podziałem na płeć i emocje (zgrupowany lub ułożony),
- wartości procentowe (nie bezwzględne).

Warunek poprawności:
Procentowe słupki każdej płci powinny sumować się do 100%. Jeśli jedna z płci ma zbyt mało kwestii (np. mniej niż 10), wypisz ostrzeżenie.

Jeśli wystąpi błąd:
Wyjaśnij, czy problem dotyczy łączenia tabel (brak wspólnej kolumny postaci), czy brakujących wartości płci.

Nie rób jeszcze:
Nie zapisuj pliku.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Wykres porównujący rozkład emocji między kobietami i mężczyznami.
- Wartości procentowe umożliwiające porównanie mimo różnej liczby kwestii.
- Wyraźne etykiety płci i kategorii emocji.

#### Pytanie interpretacyjne

Czy rozkład emocji różni się między płciami? Które emocje są „sfeminizowane" lub „zmaskulinizowane" w tym konkretnym filmie? Jak ten wynik ma się do Twoich intuicji z oglądania?

Pamiętaj o zastrzeżeniu: emocja przypisana przez model to interpretacja tekstu dialogu — nie subiektywne odczucie postaci ani intencja autorska. Wynik mówi coś o *reprezentacji*, nie o *rzeczywistości*.

---
## Etap 6: Eksport danych

Zapisujemy wyniki oceny emocji do pliku, który będzie wejściem do ostatniego modułu (synteza i prezentacja).

### Krok 6A. Zapis pliku `emocje.csv`

#### Cel i sens analityczny

`emocje.csv` zawiera wszystkie kwestie z ocenami emocji — to „wzbogacona" wersja `dialogi.csv`. W kolejnym module dołączymy te dane do sieci postaci, żeby zobaczyć, jaką emocjonalną aurę wnosi do sieci każda z postaci.

#### Prompt dla modelu

```text
Kontekst:
Masz tabelę kwestii z ocenionymi emocjami i walencjami.

Wejście:
Tabela kwestii z kolumnami: numer sceny, postać, treść kwestii, liczba słów, walencja, emocja.

Zadanie:
Zapisz tę tabelę do pliku `emocje.csv` w kodowaniu UTF-8. Następnie wygeneruj też skrócony plik `emocje_postaci.csv`, w którym każda postać ma jeden wiersz z kolumnami: postać, dominująca emocja (ta najczęstsza), średnia walencja, liczba kwestii. Na końcu pokaż, jak pobrać oba pliki z Colab.

Pokaż wynik:
- komunikat o zapisaniu obu plików i liczbie wierszy w każdym,
- 5 pierwszych wierszy `emocje_postaci.csv`,
- wskazówkę jak pobrać pliki.

Warunek poprawności:
Oba pliki powinny być niepuste. Liczba wierszy w `emocje.csv` powinna odpowiadać liczbie kwestii.

Jeśli wystąpi błąd:
Wyjaśnij, którego pliku nie udało się zapisać i dlaczego.

Nie rób już nic więcej.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Potwierdzenie zapisania plików `emocje.csv` i `emocje_postaci.csv`.
- Krótki podgląd tabeli emocjonalnych portretów postaci.
- Wskazówkę, jak pobrać pliki z Google Colab.

---
## Co dalej?

- Masz teraz `emocje.csv` (kwestie z walencją i emocją) i `emocje_postaci.csv` (emocjonalne portrety postaci).
- W kolejnym notatniku połączysz te dane z siecią postaci z pierwszego modułu i wygenerujesz **prezentację HTML** zawierającą wszystkie cztery wykresy plus interaktywną, wzbogaconą sieć.
- Jeśli łuk emocjonalny wygląda zbyt płasko albo rozkład emocji jest zdegenerowany, wróć do Kroku 3A i zmodyfikuj prompt do Gemini — iteracja jest częścią metody, nie objawem błędu.